# 🛰️ Treatment Gap Radar
### Does global antimicrobial R&D go where resistance is actually getting worse?
*Vivli AMR Data Challenge — AMR ID 00013367*

This notebook integrates **13 antimicrobial-resistance surveillance datasets** (Vivli) with the
**Global AMR R&D Hub** investment/pipeline data to build two indices and compare them:

- **Resistance Need Index (RNI)** — six indicators of resistance pressure per pathogen.
- **R&D Attention Index (RAI)** — investment, project count, and pipeline products per pathogen.

The **gap** (RNI − RAI) surfaces pathogens where resistance is high but R&D attention is low.

> *This publication or presentation is based on research using data from GSK, Innoviva Specialty
> Therapeutics, Johnson & Johnson, Paratek, Pfizer, Shionogi, Venatorx, Venus Remedies Limited,
> obtained through https://amr.vivli.org*

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
import plotly.io as pio
pio.renderers.default = "notebook_connected"   # embed figures as HTML (CDN plotly.js)
from src import viz
from src.paths import PROCESSED_DIR

# Build outputs if missing (reuses isolates_long.parquet if already harmonized).
if not (PROCESSED_DIR / "gap.parquet").exists():
    from src import pipeline
    pipeline.main(skip_harmonize=(PROCESSED_DIR / "isolates_long.parquet").exists())

long_cols = ["source", "pathogen", "antibiotic", "sir", "year", "country_iso3"]
print("Outputs ready in", PROCESSED_DIR)

Outputs ready in C:\Users\Niyel\Downloads\Data Challenge_Treatment Gap Radar (1)-20260622T211028Z-3-001\Data Challenge_Treatment Gap Radar (1)\treatment_gap_radar\data_processed


## 1 · Data harmonization
All datasets are normalized into one long table (isolate × antibiotic × S/I/R).

In [2]:
import pyarrow.parquet as pq
pf = pq.ParquetFile(PROCESSED_DIR / "isolates_long.parquet")
print(f"Harmonized rows: {pf.metadata.num_rows:,}")
src_counts = pd.read_parquet(PROCESSED_DIR / "isolates_long.parquet", columns=["source"])["source"].value_counts()
src_counts

Harmonized rows: 17,896,683


source
ATLAS          15792032
KEYSTONE        1297735
SIDERO_WT        388018
GEARS            185271
DREAM_TB          71135
SOAR_207965       47364
INNOVIVA          45288
SOAR_201910       32430
SOAR_201818       29235
PLEA_I             7315
GASAR_III           494
PLEA_II             232
SPIDAAR             134
Name: count, dtype: Int64

## 2 · Resistance indicators → Resistance Need Index
Six indicators per pathogen, normalized 0–1 and combined.

In [3]:
rni = pd.read_parquet(PROCESSED_DIR / "rni.parquet")
rni[["who", "n_isolates", "prevalence", "mic_drift", "mdr", "geo_spread", "scarcity", "pediatric", "RNI"]].round(3)

,who,n_isolates,prevalence,mic_drift,mdr,geo_spread,scarcity,pediatric,RNI
pathogen,,,,,,,,,
Enterococcus faecium,high,21831,0.374,0.006,0.564,0.955,0.700,0.330,0.843
Acinetobacter baumannii,critical,56625,0.494,0.003,0.606,0.750,0.600,0.304,0.806
Mycobacterium tuberculosis,critical,5928,0.465,0.009,0.496,1.000,1.000,NaN,0.772
Providencia stuartii,,2861,0.307,0.003,0.715,0.596,0.750,0.275,0.752
Klebsiella pneumoniae,critical,128382,0.263,0.003,0.381,0.617,0.769,0.259,0.655
Enterobacter cloacae,critical,51121,0.261,0.004,0.394,0.628,0.583,0.254,0.636
Staphylococcus epidermidis,,17698,0.235,-0.005,0.452,0.500,0.500,0.226,0.520
Escherichia coli,critical,155739,0.166,0.001,0.305,0.293,0.538,0.143,0.437
Serratia marcescens,,33394,0.202,-0.001,0.330,0.276,0.333,0.205,0.437


In [4]:
viz.indicator_heatmap(rni)

## 3 · R&D Attention Index
Text-attributed investment, project count, and pipeline products per pathogen (Global AMR R&D Hub).

In [5]:
rai = pd.read_parquet(PROCESSED_DIR / "rai.parquet")
rai[["projects", "investment", "pipeline", "therapeutics", "RAI"]].round(2)

,projects,investment,pipeline,therapeutics,RAI
pathogen,,,,,
Mycobacterium tuberculosis,2850,4.587617e+09,55,422,0.98
Staphylococcus aureus,1611,1.600958e+09,68,328,0.93
Escherichia coli,1621,1.276764e+09,44,229,0.89
Pseudomonas aeruginosa,1021,1.251556e+09,41,231,0.86
Enterobacter cloacae,671,9.287119e+08,39,132,0.82
Klebsiella pneumoniae,510,6.369834e+08,23,144,0.75
Acinetobacter baumannii,414,5.541542e+08,28,133,0.75
Salmonella enterica,810,5.859241e+08,13,50,0.73
Neisseria gonorrhoeae,258,4.290543e+08,22,39,0.70


## 4 · The Treatment Gap
RNI (x) vs RAI (y). The **red quadrant** = high resistance need, low R&D attention — the priority gaps.

In [6]:
viz.gap_quadrant()

In [7]:
gap = pd.read_parquet(PROCESSED_DIR / "gap.parquet")
print("PRIORITY TREATMENT GAPS (high need / low attention):")
gap[gap["quadrant"].str.startswith("PRIORITY")].sort_values("gap_score", ascending=False)[
    ["who", "n_isolates", "RNI", "RAI", "gap_score"]].round(3)

PRIORITY TREATMENT GAPS (high need / low attention):


,who,n_isolates,RNI,RAI,gap_score
pathogen,,,,,
Providencia stuartii,,2861,0.752,0.011,0.741
Klebsiella aerogenes,critical,21204,0.434,0.114,0.320
Proteus mirabilis,,15670,0.427,0.186,0.241
Serratia marcescens,,33394,0.437,0.222,0.215
Citrobacter freundii,,10865,0.436,0.231,0.205
Staphylococcus epidermidis,,17698,0.520,0.378,0.142


## 5 · Pathogen deep-dives
Geographic distribution and time trend for a critical pathogen–drug pair.

In [8]:
viz.resistance_choropleth('Acinetobacter baumannii', 'Meropenem')

In [9]:
viz.resistance_trend('Klebsiella pneumoniae', 'Meropenem')

## 6 · Surveillance blind spots & LMICs
Data volume by country. Blind spots (white) mark low-visibility regions — often LMICs where burden
may be high. The SPIDAAR real-world dataset (Kenya/Ghana/Uganda/Malawi) shows the point sharply.

In [10]:
viz.coverage_map()

In [11]:
long = pd.read_parquet(PROCESSED_DIR / "isolates_long.parquet",
                       columns=["source", "pathogen", "antibiotic", "sir", "country"])
sp = long[(long["source"] == "SPIDAAR") & (long["antibiotic"] == "Ceftriaxone")]
print("SPIDAAR (Sub-Saharan Africa) — % ceftriaxone-resistant (3GC-R):")
(sp.groupby("pathogen")["sir"].apply(lambda s: round((s == "R").mean()*100, 1))
   .rename("%R").reset_index())

SPIDAAR (Sub-Saharan Africa) — % ceftriaxone-resistant (3GC-R):


,pathogen,%R
0,Acinetobacter baumannii,75.0
1,Citrobacter freundii,100.0
2,Escherichia coli,82.0
3,Klebsiella aerogenes,100.0
4,Klebsiella oxytoca,100.0
5,Klebsiella pneumoniae,89.5
6,Providencia stuartii,100.0
7,Pseudomonas aeruginosa,90.9
8,Serratia marcescens,100.0


## 7 · Key findings

- **Priority treatment gaps** are dominated by **neglected Enterobacterales** — *Providencia*,
  *Klebsiella aerogenes*, *Proteus mirabilis*, *Serratia marcescens*, *Citrobacter freundii* —
  which carry substantial resistance but attract minimal R&D investment.
- The WHO-critical headline pathogens (*A. baumannii*, *K. pneumoniae*, *E. cloacae*) and **TB**
  are **well-served**: high need *and* high R&D attention.
- *S. aureus*, *P. aeruginosa* and *E. coli* receive R&D attention well above their measured need
  in this surveillance footprint.
- **Blind spots:** Sub-Saharan Africa is barely represented in the surveillance datasets, yet the
  SPIDAAR RWE cohort shows **>80% 3GC resistance** in *E. coli*/*K. pneumoniae* — high burden,
  low visibility.

### Caveats
- Only ATLAS ships native S/I/R; other datasets use a curated CLSI/TB breakpoint table (approximate,
  documented in `src/breakpoints.py`).
- R&D Hub records are tagged coarsely (mostly Gram class) and by funder geography, so RAI is an
  approximate, text-derived attribution; the gap is interpreted at pathogen level.
- Indicator weights are equal by default (`config/weights.yaml`); see sensitivity below.

In [12]:
# Sensitivity: re-rank with prevalence-weighted RNI to confirm the priority gaps are stable.
from src.rni import compute_rni
w = {"prevalence": 3, "mic_drift": 1, "mdr": 1, "geo_spread": 1, "scarcity": 1, "pediatric": 1}
alt = compute_rni(weights=w, save=False).sort_values("RNI", ascending=False)
alt[["who", "RNI"]].head(8).round(3)

,who,RNI
pathogen,,
Acinetobacter baumannii,critical,0.855
Enterococcus faecium,high,0.820
Mycobacterium tuberculosis,critical,0.814
Providencia stuartii,,0.716
Klebsiella pneumoniae,critical,0.621
Enterobacter cloacae,critical,0.605
Staphylococcus epidermidis,,0.504
Klebsiella aerogenes,critical,0.428
